# PFS AG Log Explorer

Interactive exploration of PFS Auto-Guider actor log files.

**Run all cells** to launch the app, or serve it with:
```
panel serve ag_explorer.ipynb --show
```

Controls:
- **Log directory** — type any path; file list refreshes automatically
- **Log file** — newest first, with file sizes shown
- **Design** — filter to a single design window
- **Panels** — toggle individual plot panels on/off
- **Follow / Tail** — stream the file live; set the refresh interval in seconds


In [ ]:
import importlib.util
import sys
from bisect import bisect_right
from collections import defaultdict
from datetime import datetime, timedelta
from pathlib import Path

import panel as pn
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pn.extension('plotly', sizing_mode='stretch_width')


In [ ]:
# Dynamically import parsers and DataStore from ag_dashboard.py,
# which lives alongside this notebook.
_HERE = Path().resolve()

_DASH = str(_HERE / 'ag_dashboard.py')
spec = importlib.util.spec_from_file_location('ag_dashboard', _DASH)
_mod = importlib.util.module_from_spec(spec)
sys.modules['ag_dashboard'] = _mod
spec.loader.exec_module(_mod)

from ag_dashboard import (  # noqa: E402
    DataStore,
    HST, _night_bounds_from_store,
    parse_line,
)


In [ ]:
_record_cache: dict[str, dict] = {}


def _fmt_size(n: int) -> str:
    if n < 1024:       return f'{n} B'
    if n < 1_048_576:  return f'{n/1024:.1f} KB'
    return f'{n/1_048_576:.1f} MB'


def _parse_log(path: str) -> dict[str, list]:
    """Parse entire file, caching results so re-selecting is instant."""
    if path not in _record_cache:
        state: dict = {}
        records: dict[str, list] = defaultdict(list)
        with open(path, errors='replace') as f:
            for line in f:
                for rtype, rec in parse_line(line.rstrip('\n'), state):
                    records[rtype].append(rec)
        _record_cache[path] = dict(records)
    return _record_cache[path]


def _parse_from_pos(path: str, pos: int, state: dict) -> tuple[dict[str, list], int]:
    """Read from byte offset *pos*, return (new_records, new_pos)."""
    new_records: dict[str, list] = defaultdict(list)
    with open(path, errors='replace') as f:
        f.seek(pos)
        for line in f:
            for rtype, rec in parse_line(line.rstrip('\n'), state):
                new_records[rtype].append(rec)
        new_pos = f.tell()
    return dict(new_records), new_pos


def _make_store(records: dict[str, list], t_start=None, t_end=None) -> DataStore:
    store = DataStore(window_minutes=None)
    for rtype, recs in records.items():
        for rec in recs:
            if (t_start is None or rec.t >= t_start) and \
               (t_end   is None or rec.t <  t_end):
                store.push(rtype, rec)
    return store


def _last_t(records: dict[str, list]):
    last = None
    for recs in records.values():
        if recs and (last is None or recs[-1].t > last):
            last = recs[-1].t
    return last


def _context_arrays(tel_recs: list, store: 'DataStore') -> list:
    """For each tel_axes record return [design_str, visit_str, frame_str]."""
    designs = store.snapshot('design_changes')
    visits  = store.snapshot('visit_changes')
    guides  = store.snapshot('guide')
    d_times = [r.t for r in designs]
    v_times = [r.t for r in visits]
    g_times = [r.t for r in guides]
    out = []
    for r in tel_recs:
        di = bisect_right(d_times, r.t) - 1
        vi = bisect_right(v_times, r.t) - 1
        gi = bisect_right(g_times, r.t) - 1
        out.append([
            f'd…{str(designs[di].design_id)[-8:]}' if di >= 0 else '—',
            f'v{visits[vi].visit_id}'                  if vi >= 0 else '—',
            str(guides[gi].exp_id)                     if gi >= 0 else '—',
        ])
    return out


def _guide_context_arrays(guide_recs: list, store: 'DataStore') -> list:
    """For each guide record return [design_str, visit_str, frame_str]."""
    designs = store.snapshot('design_changes')
    visits  = store.snapshot('visit_changes')
    d_times = [r.t for r in designs]
    v_times = [r.t for r in visits]
    out = []
    for r in guide_recs:
        di = bisect_right(d_times, r.t) - 1
        vi = bisect_right(v_times, r.t) - 1
        out.append([
            f'd…{str(designs[di].design_id)[-8:]}' if di >= 0 else '—',
            f'v{visits[vi].visit_id}'                  if vi >= 0 else '—',
            str(r.exp_id),
        ])
    return out


def _hst(t):
    """Convert timezone-aware datetime to naive HST for Plotly display."""
    return t.astimezone(HST).replace(tzinfo=None)


def _build_design_labels(designs: list) -> list[str]:
    labels = ['All designs']
    for i, r in enumerate(designs):
        t0 = r.t.astimezone(HST).strftime('%H:%M')
        t1_obj = designs[i + 1].t if i + 1 < len(designs) else None
        t1 = t1_obj.astimezone(HST).strftime('%H:%M') if t1_obj else 'end'
        labels.append(f'd…{str(r.design_id)[-8:]}  {t0}–{t1} HST')
    return labels


def _has_data(store: DataStore) -> bool:
    return any(store.snapshot(k) for k in ('guide', 'focus', 'star_stats', 'camera_counts', 'tel_axes'))


In [ ]:
PANEL_NAMES = ['Guide Offsets', 'Focus', 'Star Quality', 'Camera Counts', 'Telescope']

_PANEL_TITLES = {
    'Guide Offsets':  'Guide offsets',
    'Focus':          'Focus offsets & scale',
    'Star Quality':   'Star quality / seeing proxy',
    'Camera Counts':  'Per-camera object counts',
    'Telescope':      'Telescope position',
}
_GUIDE_COLORS = ['#1f77b4', '#ff7f0e', '#d62728', '#9467bd']
_Z_COLORS     = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#17becf']
_CAM_COLORS   = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']


def _event_shapes_and_annotations(store: DataStore) -> tuple[list, list]:
    shapes, annotations = [], []
    for rec in store.snapshot('visit_changes'):
        x = _hst(rec.t).isoformat()
        shapes.append(dict(
            type='line', xref='paper' if False else 'x', yref='paper',
            x0=x, x1=x, y0=0, y1=1,
            line=dict(color='royalblue', width=1.4, dash='solid'),
            opacity=0.3, layer='below',
        ))
        annotations.append(dict(
            x=x, y=0.02, xref='x', yref='paper',
            text=f'v{rec.visit_id}', textangle=-90,
            showarrow=False, font=dict(size=9, color='royalblue'),
            xanchor='left', yanchor='bottom',
        ))
    for rec in store.snapshot('design_changes'):
        x = _hst(rec.t).isoformat()
        shapes.append(dict(
            type='line', xref='x', yref='paper',
            x0=x, x1=x, y0=0, y1=1,
            line=dict(color='darkorange', width=1.4, dash='dash'),
            opacity=0.3, layer='below',
        ))
        annotations.append(dict(
            x=x, y=0.02, xref='x', yref='paper',
            text=f'd{str(rec.design_id)[-8:]}', textangle=-90,
            showarrow=False, font=dict(size=9, color='darkorange'),
            xanchor='right', yanchor='bottom',
        ))
    return shapes, annotations



def _build_figure(store: DataStore, active_panels: list[str], x_start, x_end) -> go.Figure:
    import math
    n = len(active_panels)
    if n == 0:
        return go.Figure()

    GAP = 0.03

    def _dom(i):
        """Y-axis domain [bottom, top] for panel i (0 = topmost panel)."""
        h = (1.0 - (n - 1) * GAP) / n
        return [round(1.0 - (i + 1) * h - i * GAP, 4),
                round(1.0 - i * h - i * GAP, 4)]

    def _yaxes(i):
        """Return (p_ref, s_ref, p_key, s_key) for panel i."""
        p = 2 * i + 1
        s = 2 * i + 2
        p_ref = 'y' if p == 1 else f'y{p}'
        s_ref = f'y{s}'
        p_key = 'yaxis' if p == 1 else f'yaxis{p}'
        s_key = f'yaxis{s}'
        return p_ref, s_ref, p_key, s_key

    fig = go.Figure()
    layout_updates = {}

    for i, panel in enumerate(active_panels):
        dom = _dom(i)
        p_ref, s_ref, p_key, s_key = _yaxes(i)
        lg = f'p{i}'

        layout_updates[p_key] = dict(domain=dom)
        layout_updates[s_key] = dict(overlaying=p_ref, side='right')

        if panel == 'Guide Offsets':
            recs = store.snapshot('guide')
            if recs:
                ts  = [_hst(r.t) for r in recs]
                ctx = _guide_context_arrays(recs, store)
                for first, ((attr, label), color) in enumerate(zip(
                    [('dra', 'dRA'), ('ddec', 'dDec'), ('daz', 'dAz'), ('del_', 'dEl')],
                    _GUIDE_COLORS,
                )):
                    if first == 0:
                        ht = (
                            '<b>%{x|%H:%M:%S} HST</b><br>'
                            'dRA: %{y:.4f} arcsec<br>'
                            '──────────────────<br>'
                            'Design: %{customdata[0]}<br>'
                            'Visit:  %{customdata[1]}<br>'
                            'Frame:  %{customdata[2]}'
                            '<extra></extra>'
                        )
                    else:
                        ht = f'{label}: %{{y:.4f}} arcsec<extra></extra>'
                    fig.add_trace(go.Scatter(
                        x=ts, y=[getattr(r, attr) for r in recs],
                        name=label, mode='lines+markers',
                        marker=dict(size=4), line=dict(color=color, width=1.2),
                        legendgroup=lg, xaxis='x', yaxis=p_ref,
                        customdata=ctx, hovertemplate=ht,
                    ))
                fig.add_trace(go.Scatter(
                    x=ts, y=[r.dinr for r in recs],
                    name='dInR', mode='lines+markers',
                    marker=dict(size=4), line=dict(color='#2ca02c', width=1.2, dash='dash'),
                    legendgroup=lg, xaxis='x', yaxis=s_ref,
                    hovertemplate='dInR: %{y:.4f} arcsec<extra></extra>',
                ))
            layout_updates[p_key].update(title_text='arcsec')
            layout_updates[s_key].update(title_text='dInR (arcsec)',
                                         tickfont=dict(color='#2ca02c'),
                                         title_font=dict(color='#2ca02c'))

        elif panel == 'Focus':
            g = store.snapshot('guide')
            f = store.snapshot('focus')
            if g:
                ts = [_hst(r.t) for r in g]
                fig.add_trace(go.Scatter(
                    x=ts, y=[r.dfocus for r in g],
                    name='dFocus', mode='lines+markers',
                    marker=dict(size=4, color='black'), line=dict(color='black', width=1.2),
                    legendgroup=lg, xaxis='x', yaxis=p_ref,
                ))
                fig.add_trace(go.Scatter(
                    x=ts, y=[r.dscale * 1e6 for r in g],
                    name='dScale \u00d710\u207b\u2076', mode='lines+markers',
                    marker=dict(size=4, color='gray'), line=dict(color='gray', width=1.2),
                    legendgroup=lg, xaxis='x', yaxis=s_ref,
                ))
            if f:
                ts = [_hst(r.t) for r in f]
                for attr, color, label in zip(
                    ['z1', 'z2', 'z3', 'z4', 'z5', 'z6'],
                    _Z_COLORS,
                    ['Z1', 'Z2', 'Z3', 'Z4', 'Z5', 'Z6'],
                ):
                    vals = [getattr(r, attr) for r in f]
                    offline = all(math.isnan(v) for v in vals)
                    fig.add_trace(go.Scatter(
                        x=ts, y=vals,
                        name=f'{label} (offline)' if offline else label,
                        mode='markers',
                        marker=dict(size=4, color='lightgrey' if offline else color),
                        legendgroup=lg, xaxis='x', yaxis=p_ref,
                    ))
            layout_updates[p_key].update(title_text='Focus offset (mm)')
            layout_updates[s_key].update(title_text='dScale (\u00d710\u207b\u2076)',
                                         tickfont=dict(color='gray'),
                                         title_font=dict(color='gray'))

        elif panel == 'Star Quality':
            ss = store.snapshot('star_stats')
            if ss:
                ts = [_hst(r.t) for r in ss]
                fig.add_trace(go.Scatter(
                    x=ts, y=[r.size for r in ss],
                    name='PSF size (px)', mode='lines+markers',
                    marker=dict(size=4), line=dict(color='#1f77b4', width=1.2),
                    legendgroup=lg, xaxis='x', yaxis=p_ref,
                ))
                fig.add_trace(go.Scatter(
                    x=ts, y=[r.peak for r in ss],
                    name='peak ADU', mode='lines+markers',
                    marker=dict(size=4), line=dict(color='#ff7f0e', width=1.2, dash='dash'),
                    legendgroup=lg, xaxis='x', yaxis=s_ref,
                ))
            layout_updates[p_key].update(title_text='PSF size (px)',
                                         tickfont=dict(color='#1f77b4'),
                                         title_font=dict(color='#1f77b4'))
            layout_updates[s_key].update(title_text='peak ADU (log)', type='log',
                                         tickfont=dict(color='#ff7f0e'),
                                         title_font=dict(color='#ff7f0e'))

        elif panel == 'Camera Counts':
            recs = store.snapshot('camera_counts')
            if recs:
                ts = [_hst(r.t) for r in recs]
                for j, color in enumerate(_CAM_COLORS):
                    vals = [r.counts[j] if r.counts[j] is not None else None for r in recs]
                    fig.add_trace(go.Scatter(
                        x=ts, y=vals,
                        name=f'AGC{j + 1}', mode='lines+markers',
                        marker=dict(size=3), line=dict(color=color, width=1),
                        legendgroup=lg, xaxis='x', yaxis=p_ref,
                    ))
            layout_updates[p_key].update(title_text='detected sources')

        elif panel == 'Telescope':
            recs = store.snapshot('tel_axes')
            if recs:
                ts = [_hst(r.t) for r in recs]
                for attr, label, color, sec in [
                    ('az',      'Az (deg)',  '#1f77b4', False),
                    ('el',      'El (deg)',  '#ff7f0e', False),
                    ('airmass', 'Airmass',   '#2ca02c', True),
                ]:
                    fig.add_trace(go.Scatter(
                        x=ts, y=[getattr(r, attr) for r in recs],
                        name=label, mode='lines+markers',
                        marker=dict(size=4),
                        line=dict(color=color, width=1.2, dash='dash' if sec else 'solid'),
                        legendgroup=lg, xaxis='x', yaxis=s_ref if sec else p_ref,
                    ))
            layout_updates[p_key].update(title_text='Az / El (deg)')
            layout_updates[s_key].update(title_text='Airmass',
                                         tickfont=dict(color='#2ca02c'),
                                         title_font=dict(color='#2ca02c'))

    # ── Align zeros on dual y-axis panels ─────────────────────────────────────
    for i, panel in enumerate(active_panels):
        p_ref, s_ref, p_key, s_key = _yaxes(i)
        if layout_updates.get(s_key, {}).get('type') == 'log':
            continue  # log scale has no zero to align
        p_traces = [t for t in fig.data if getattr(t, 'yaxis', None) == p_ref]
        s_traces = [t for t in fig.data if getattr(t, 'yaxis', None) == s_ref]
        if not p_traces or not s_traces:
            continue
        def _finite(traces):
            return [float(v) for t in traces for v in (t.y or [])
                    if v is not None and not math.isnan(float(v))]
        pv, sv = _finite(p_traces), _finite(s_traces)
        if not pv or not sv:
            continue
        lo_p, hi_p = min(pv), max(pv)
        lo_s, hi_s = min(sv), max(sv)
        # 5 % padding
        pad_p = max(abs(hi_p - lo_p) * 0.05, 1e-9)
        pad_s = max(abs(hi_s - lo_s) * 0.05, 1e-9)
        lo_p -= pad_p; hi_p += pad_p
        lo_s -= pad_s; hi_s += pad_s
        if lo_p >= 0 or hi_p <= 0 or lo_s >= 0 or hi_s <= 0:
            continue  # zero outside a range — nothing to align
        # Pick the fraction f in [0,1] where zero must sit so both datasets fit
        f_p = -lo_p / (hi_p - lo_p)
        f_s = -lo_s / (hi_s - lo_s)
        f = max(f_p, f_s)
        r = f / (1.0 - f)  # ratio |lo| / hi at aligned zero
        def _align(lo, hi):
            if r * hi >= -lo:      # negative side is the constraint
                return -r * hi, hi
            else:                   # positive side is the constraint
                return lo, -lo / r
        layout_updates[p_key]['range'] = list(_align(lo_p, hi_p))
        layout_updates[s_key]['range'] = list(_align(lo_s, hi_s))

    shapes, event_annotations = _event_shapes_and_annotations(store)

    title_annotations = [
        dict(
            x=0, y=_dom(i)[1],
            xref='paper', yref='paper',
            text=f'<b>{_PANEL_TITLES[p]}</b>',
            xanchor='left', yanchor='bottom',
            showarrow=False, font=dict(size=11),
        )
        for i, p in enumerate(active_panels)
    ]

    fig.update_layout(
        **layout_updates,
        xaxis=dict(
            domain=[0, 1],
            range=[_hst(x_start).isoformat(), _hst(x_end).isoformat()],
            title_text='Time (HST)',
            showspikes=True,
            spikemode='across',
            spikesnap='cursor',
            spikecolor='rgba(100,100,100,0.5)',
            spikethickness=1,
            spikedash='solid',
        ),
        height=max(300, 260 * n),
        hovermode='x',
        legend=dict(orientation='v', x=1.08, y=1, tracegroupgap=12),
        margin=dict(l=60, r=160, t=40, b=60),
        shapes=shapes,
        annotations=event_annotations + title_annotations,
    )
    return fig


In [ ]:
# ── Widgets ───────────────────────────────────────────────────────────────────

_LOGS_DEFAULT = str(_HERE / 'logs')

dir_input = pn.widgets.TextInput(
    name='Log directory', value=_LOGS_DEFAULT, width=280,
)
file_select = pn.widgets.Select(name='Log file', width=280)
design_select = pn.widgets.Select(name='Design', width=280)
panels_check = pn.widgets.CheckBoxGroup(
    name='Panels', options=PANEL_NAMES, value=PANEL_NAMES,
)
follow_toggle = pn.widgets.Toggle(
    name='▶ Follow / Tail', button_type='success', width=150,
)
interval_input = pn.widgets.IntInput(
    name='Refresh (s)', value=5, start=1, end=300, step=1, width=100,
)
tail_window_select = pn.widgets.Select(
    name='Tail window',
    options={
        'All data': 'all',
        'Last 5 min': '5m',
        'Last 10 min': '10m',
        'Last 30 min': '30m',
        'Current visit': 'visit',
        'Current design': 'design',
    },
    value='all',
    width=150,
)
status_md = pn.pane.Markdown('', width=280)
plot_pane = pn.pane.Plotly(go.Figure(), sizing_mode='stretch_width', min_height=400)

# ── Internal state ────────────────────────────────────────────────────────────

_record_cache_local: dict[str, dict] = {}
_busy = [False]
_follow_cb = [None]
_tail_state: dict = {}
_tail_pos = [0]
_tail_records: dict[str, list] = {}


# ── Helpers ───────────────────────────────────────────────────────────────────

def _log_path():
    d = Path(dir_input.value.strip())
    v = file_select.value
    return d / v if v else None


def _refresh_file_list(directory: str):
    p = Path(directory.strip())
    paths = sorted(p.glob('*.log'), reverse=True) if p.is_dir() else []
    opts = {f'{pp.name}  ({_fmt_size(pp.stat().st_size)})': pp.name for pp in paths}
    file_select.param.update(options=opts, value=list(opts.values())[0] if opts else None)


def _refresh_design_list(records: dict):
    designs = records.get('design_changes', [])
    design_select.param.update(
        options=_build_design_labels(designs),
        value='All designs',
    )


def _current_records():
    """Return records to plot: tail buffer when following, cache otherwise."""
    if follow_toggle.value and _tail_records:
        return _tail_records
    p = _log_path()
    return _parse_log(str(p)) if p and p.exists() else {}


def _tail_window_bounds(records: dict):
    """Return (t_start, t_end) for the active tail window, or (None, None) for all data."""
    window = tail_window_select.value
    if window == '5m':
        return datetime.now(tz=HST) - timedelta(minutes=5), None
    if window == '10m':
        return datetime.now(tz=HST) - timedelta(minutes=10), None
    if window == '30m':
        return datetime.now(tz=HST) - timedelta(minutes=30), None
    if window == 'visit':
        visits = records.get('visit_changes', [])
        return (visits[-1].t, None) if visits else (None, None)
    if window == 'design':
        designs = records.get('design_changes', [])
        return (designs[-1].t, None) if designs else (None, None)
    return None, None  # 'all'


def _render():
    p = _log_path()
    if not p:
        plot_pane.object = go.Figure()
        status_md.object = '⚠️ No log file selected.'
        return
    try:
        records = _current_records()
        designs = records.get('design_changes', [])
        labels  = list(design_select.options) or ['All designs']
        sel     = design_select.value or 'All designs'

        if follow_toggle.value and tail_window_select.value != 'all':
            t_start, t_end = _tail_window_bounds(records)
            if t_start is not None:
                store = _make_store(records, t_start, t_end)
                x_start = t_start
                x_end = t_end or _last_t(records) or t_start
            else:
                store = _make_store(records)
                x_start, x_end = _night_bounds_from_store(store)
        elif sel == 'All designs':
            store = _make_store(records)
            x_start, x_end = _night_bounds_from_store(store)
        else:
            d_idx   = max(0, labels.index(sel) - 1)
            t_start = designs[d_idx].t
            t_end   = designs[d_idx + 1].t if d_idx + 1 < len(designs) else None
            store   = _make_store(records, t_start, t_end)
            x_start = t_start
            x_end   = t_end or _last_t(records) or t_start

        if not _has_data(store):
            plot_pane.object = go.Figure()
            total = sum(len(v) for v in records.values())
            if total == 0:
                status_md.object = f'⚠️ **{p.name}** — empty or unrecognised format.'
            else:
                status_md.object = (
                    f'⚠️ **{p.name}** — {total} lines parsed, no AG guiding data.'
                )
            return

        fig = _build_figure(store, list(panels_check.value), x_start, x_end)
        plot_pane.object = fig
        g = len(store.snapshot('guide'))
        v = len(store.snapshot('visit_changes'))
        d = len(store.snapshot('design_changes'))
        status_md.object = f'**Exposures:** {g} · **Visits:** {v} · **Designs:** {d}'

    except Exception as exc:
        import traceback
        plot_pane.object = go.Figure()
        status_md.object = f'❌ `{type(exc).__name__}: {exc}`'
        traceback.print_exc()


# ── Tail mode ─────────────────────────────────────────────────────────────────

def _stop_follow():
    if _follow_cb[0] is not None:
        try:
            _follow_cb[0].stop()
        except Exception:
            pass
        _follow_cb[0] = None


def _init_tail():
    global _tail_records, _tail_state
    p = _log_path()
    if not p or not p.exists():
        return
    records = _parse_log(str(p))
    _tail_records = {k: list(v) for k, v in records.items()}
    _tail_state = {}
    with open(str(p), errors='replace') as f:
        f.seek(0, 2)
        _tail_pos[0] = f.tell()


def _tail_tick():
    p = _log_path()
    if not p or not p.exists():
        return
    try:
        new_recs, new_pos = _parse_from_pos(str(p), _tail_pos[0], _tail_state)
        _tail_pos[0] = new_pos
        if new_recs:
            for rtype, recs in new_recs.items():
                _tail_records.setdefault(rtype, []).extend(recs)
            if 'design_changes' in new_recs:
                _busy[0] = True
                design_select.options = _build_design_labels(
                    _tail_records.get('design_changes', [])
                )
                _busy[0] = False
            _render()
    except Exception as exc:
        status_md.object = f'❌ Tail error: `{exc}`'


# ── Widget callbacks ──────────────────────────────────────────────────────────

def _load_file():
    """Parse the selected file, refresh the design dropdown, and render.
    Called explicitly so directory changes always trigger a full reload even
    when file_select.value hasn't changed."""
    v = file_select.value
    if not v:
        plot_pane.object = go.Figure()
        status_md.object = '⚠️ No log file selected.'
        return
    p = Path(dir_input.value.strip()) / v
    if not p.exists():
        status_md.object = f'⚠️ Not found: `{p}`'
        return
    _busy[0] = True
    records = _parse_log(str(p))
    _refresh_design_list(records)
    _busy[0] = False
    _render()


@pn.depends(dir_input.param.value, watch=True)
def _on_dir(value):
    _stop_follow()
    follow_toggle.value = False
    _busy[0] = True          # suppress _on_file watcher during list refresh
    _refresh_file_list(value)
    _busy[0] = False
    _load_file()             # explicit reload regardless of whether value changed


@pn.depends(file_select.param.value, watch=True)
def _on_file(value):
    if _busy[0]:
        return               # _on_dir is driving; it will call _load_file()
    _stop_follow()
    follow_toggle.value = False
    _load_file()


@pn.depends(design_select.param.value, watch=True)
def _on_design(value):
    if not _busy[0]:
        _render()


@pn.depends(panels_check.param.value, watch=True)
def _on_panels(value):
    _render()


@pn.depends(tail_window_select.param.value, watch=True)
def _on_tail_window(value):
    if follow_toggle.value:
        _render()


@pn.depends(follow_toggle.param.value, watch=True)
def _on_follow(active):
    _stop_follow()
    if active:
        follow_toggle.name = '⏸ Following…'
        _init_tail()
        _tail_tick()
        _follow_cb[0] = pn.state.add_periodic_callback(
            _tail_tick, period=interval_input.value * 1000,
        )
    else:
        follow_toggle.name = '▶ Follow / Tail'
        _render()


# ── Layout ────────────────────────────────────────────────────────────────────

sidebar = pn.Column(
    pn.pane.Markdown('### Controls', margin=(5, 5, 0, 5)),
    dir_input,
    file_select,
    design_select,
    pn.layout.Divider(),
    pn.pane.Markdown('**Panels**', margin=(0, 5)),
    panels_check,
    pn.layout.Divider(),
    pn.Row(follow_toggle, interval_input),
    tail_window_select,
    pn.layout.Divider(),
    status_md,
    width=310,
    sizing_mode='fixed',
)

app = pn.Row(
    sidebar,
    pn.Column(plot_pane, sizing_mode='stretch_both'),
    sizing_mode='stretch_both',
)

app.servable()

# Initial population (runs in notebook; panel serve calls servable() instead)
_refresh_file_list(dir_input.value)
